# 09.04 -- Quantisation: Running Large Models in Less Memory

**Module:** 09 -- LLM Inference Engineering  
**Notebook:** 4 of 5  
**Prerequisites:** 09.01 (KV Cache)

---

## Learning Objectives

1. Understand why quantisation is necessary and what limits the minimum viable precision
2. Implement uniform INT8 and INT4 quantisation from scratch with calibration
3. Understand NF4 (Normal Float 4) and why it outperforms uniform quantisation for weights
4. Measure the memory-accuracy trade-off across precision levels on a real model
5. Apply post-training quantisation using bitsandbytes and compare to GPTQ

---

## 1. Why Quantise?

A 7B parameter model stored in 32-bit float (fp32) occupies 28 GB of GPU memory. The same model in 16-bit (fp16/bf16) requires 14 GB. With 8-bit quantisation it requires 7 GB, and with 4-bit it requires only 3.5 GB -- fitting on a single consumer GPU.

Quantisation replaces high-precision floating point values with lower-precision integers or custom float formats. The central question is how much accuracy is lost in exchange for this memory saving.

```
Format    Bits   Memory (7B)   Typical accuracy loss
------------------------------------------------
fp32       32     28 GB         0% (reference)
bf16       16     14 GB         <0.1%  (negligible)
fp8         8      7 GB         0.1-0.5%
int8        8      7 GB         0.5-2%
int4        4      3.5 GB       2-5%
NF4         4      3.5 GB       1-3%  (better than int4 for weights)
int2        2      1.75 GB      >10%  (rarely usable)
```

---

## 2. The Quantisation Mapping

Quantisation maps a range of floating point values to a finite set of integer levels:

```
Quantise:   q = round(x / scale) + zero_point
Dequantise: x_approx = (q - zero_point) * scale

where:
  scale      = (x_max - x_min) / (2^bits - 1)
  zero_point = -round(x_min / scale)  [for asymmetric]
```

The quantisation error is bounded by scale/2 (half the step size between levels). A finer grid (more bits) means smaller steps and smaller error.

In [ ]:
# Environment setup.
#
# bitsandbytes provides INT8 and NF4 quantisation kernels that run
# on CUDA GPUs. It is the library used by HuggingFace's load_in_8bit
# and load_in_4bit features and by QLoRA (Notebook 07.02).
# We use it here for the production quantisation section.
# The from-scratch implementations in sections 3-5 use only NumPy.

!pip install transformers bitsandbytes accelerate matplotlib numpy scipy -q

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 3. Uniform Quantisation From Scratch

In [ ]:
# Uniform quantisation: symmetric and asymmetric variants.
#
# Symmetric quantisation maps the range [-x_max, x_max] to [-2^(b-1), 2^(b-1)-1].
# The zero point is always 0, which simplifies the dequantisation multiply:
#   x_approx = q * scale
# Symmetric quantisation is preferred for weights because weight distributions
# are typically symmetric around zero after training.
#
# Asymmetric quantisation maps [x_min, x_max] to [0, 2^b - 1].
# It uses a non-zero zero point to shift the representation range:
#   x_approx = (q - zero_point) * scale
# Asymmetric quantisation is preferred for activations, which are often
# non-symmetric (e.g., ReLU outputs are always >= 0).
#
# Both variants use per-tensor scale factors by default. Per-channel
# quantisation (one scale per output channel) preserves accuracy better
# at the cost of slightly higher memory overhead for the scale tensors.

class UniformQuantiser:
    """
    Uniform quantisation and dequantisation with configurable bit width.

    Supports both symmetric (zero_point=0) and asymmetric quantisation.
    """

    def __init__(self, bits: int = 8, symmetric: bool = True, per_channel: bool = False):
        self.bits        = bits
        self.symmetric   = symmetric
        self.per_channel = per_channel

        if symmetric:
            self.qmin = -(2 ** (bits - 1))
            self.qmax =  (2 ** (bits - 1)) - 1
        else:
            self.qmin = 0
            self.qmax = (2 ** bits) - 1

    def compute_scale_zp(
        self,
        tensor: torch.Tensor,   # the tensor to quantise
        dim: int = 0,           # channel dimension (only used when per_channel=True)
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Compute scale and zero_point from the tensor's min/max range.

        The scale maps the full float range to the integer range.
        The zero_point shifts the integer range to align with zero.
        """
        if self.per_channel:
            # Reduce over all dims except the channel dim
            reduce_dims = [i for i in range(tensor.dim()) if i != dim]
            x_min = tensor.amin(dim=reduce_dims, keepdim=True)
            x_max = tensor.amax(dim=reduce_dims, keepdim=True)
        else:
            x_min = tensor.min()
            x_max = tensor.max()

        if self.symmetric:
            # Use the larger absolute value so the range is symmetric
            x_abs_max = torch.maximum(x_min.abs(), x_max.abs())
            scale = x_abs_max / self.qmax
            scale = torch.clamp(scale, min=1e-8)  # avoid divide-by-zero
            zero_point = torch.zeros_like(scale)
        else:
            scale = (x_max - x_min) / (self.qmax - self.qmin)
            scale = torch.clamp(scale, min=1e-8)
            zero_point = torch.round(-x_min / scale + self.qmin)
            zero_point = zero_point.clamp(self.qmin, self.qmax)

        return scale.float(), zero_point.float()

    def quantise(
        self,
        tensor:     torch.Tensor,
        scale:      torch.Tensor,
        zero_point: torch.Tensor,
    ) -> torch.Tensor:
        """Map floating-point values to integers in [qmin, qmax]."""
        q = torch.round(tensor / scale + zero_point)
        return q.clamp(self.qmin, self.qmax).to(torch.int8 if self.bits == 8 else torch.int32)

    def dequantise(
        self,
        q:          torch.Tensor,
        scale:      torch.Tensor,
        zero_point: torch.Tensor,
    ) -> torch.Tensor:
        """Reconstruct float values from integer codes."""
        return (q.float() - zero_point) * scale

    def quantise_dequantise(
        self,
        tensor: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Full quantise-dequantise round-trip. Returns (approx, scale, zero_point)."""
        scale, zp = self.compute_scale_zp(tensor)
        q          = self.quantise(tensor, scale, zp)
        approx     = self.dequantise(q, scale, zp)
        return approx, scale, zp


# Test quantisation error at different bit widths
# Use normally distributed weights, typical of a trained transformer layer
weights = torch.randn(256, 256)  # typical weight matrix size

print(f"Original weight statistics:")
print(f"  min={weights.min():.3f}  max={weights.max():.3f}  ")
print(f"  mean={weights.mean():.4f}  std={weights.std():.3f}")
print()
print(f"{'Bits':>6} {'Symmetric':>12} {'Mean Abs Error':>16} {'Max Abs Error':>14} {'SNR (dB)':>10}")
print("-" * 62)

for bits in [8, 4, 3, 2]:
    for sym in [True, False]:
        q = UniformQuantiser(bits=bits, symmetric=sym)
        approx, _, _ = q.quantise_dequantise(weights)
        err = (weights - approx).abs()
        signal_power = weights.pow(2).mean()
        noise_power  = (weights - approx).pow(2).mean()
        snr_db = 10 * torch.log10(signal_power / noise_power).item()
        print(f"{bits:>6}   {'Yes' if sym else 'No':>10}   "
              f"{err.mean().item():>14.6f}   {err.max().item():>14.6f}   {snr_db:>8.1f}")

In [ ]:
# Visualise quantisation error across bit widths.
#
# We show three things for each bit width:
#   1. The quantisation levels (where values are rounded to)
#   2. The quantisation error as a function of the original value
#   3. The error distribution (histogram)
#
# The sawtooth pattern in the error plot is characteristic of uniform
# quantisation: error is highest midway between quantisation levels
# and zero exactly at the levels. This pattern is the same regardless
# of bit width; only the period (step size) changes.
#
# The key insight from these plots: reducing from 8 to 4 bits roughly
# doubles the step size and quadruples the mean squared quantisation error.
# But the actual accuracy loss in a transformer depends on which tensors
# are quantised and how sensitive the model is to errors in each layer.

x = torch.linspace(-3, 3, 500)
fig, axes = plt.subplots(3, 3, figsize=(15, 11))

bit_configs = [(8, True), (4, True), (3, True)]
colors_map  = ['#1f77b4', '#ff7f0e', '#2ca02c']

for col, ((bits, sym), color) in enumerate(zip(bit_configs, colors_map)):
    q = UniformQuantiser(bits=bits, symmetric=sym)
    approx, scale, zp = q.quantise_dequantise(x)
    error = (x - approx)

    # Row 0: original vs quantised
    ax = axes[0, col]
    ax.plot(x.numpy(), x.numpy(), 'k--', alpha=0.4, linewidth=1, label='Original')
    ax.step(x.numpy(), approx.numpy(), color=color, linewidth=1.5, label=f'Quantised ({bits}-bit)')
    ax.set_title(f'{bits}-bit symmetric\nStep size = {scale.item():.4f}', fontsize=11)
    ax.set_xlabel('Original value', fontsize=9)
    ax.set_ylabel('Quantised value', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Row 1: quantisation error
    ax = axes[1, col]
    ax.plot(x.numpy(), error.numpy(), color=color, linewidth=1.5)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Original value', fontsize=9)
    ax.set_ylabel('Quantisation error', fontsize=9)
    ax.set_title(f'Error (sawtooth, max={error.abs().max().item():.4f})', fontsize=11)
    ax.grid(alpha=0.3)

    # Row 2: error distribution
    ax = axes[2, col]
    # Apply to normally distributed weights to show realistic error distribution
    w = torch.randn(10000)
    approx_w, _, _ = q.quantise_dequantise(w)
    err_w = (w - approx_w).numpy()
    ax.hist(err_w, bins=50, color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('Quantisation error', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.set_title(f'Error distribution (normal weights)\nRMSE={np.sqrt(np.mean(err_w**2)):.5f}', fontsize=11)
    ax.grid(alpha=0.3)

plt.suptitle('Uniform Quantisation: Error Analysis Across Bit Widths', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/09_quantisation_error.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. NF4: Normal Float 4 From Scratch

In [ ]:
# NF4 (Normal Float 4) quantisation, as used in QLoRA and bitsandbytes.
#
# Uniform quantisation is suboptimal when the data distribution is known.
# Transformer weights are approximately normally distributed after training.
# Uniform 4-bit quantisation spaces levels evenly across the full range,
# which wastes representational capacity on the tails where almost no
# weight values live.
#
# NF4 places levels at the quantiles of a standard normal distribution:
# more levels near the centre (where most weights are) and fewer at the
# tails. This minimises expected quantisation error for normally
# distributed inputs, at the cost of being suboptimal for other distributions.
#
# The NF4 levels are fixed constants -- they do not depend on the data.
# Only the block-wise scale factor (see below) is computed per-tensor.
# This makes dequantisation cheap: one table lookup + one multiply.
#
# The exact NF4 levels come from Table 1 of the QLoRA paper (Dettmers et al., 2023).

from scipy.stats import norm as scipy_norm

def compute_nf4_levels(n_levels: int = 16) -> np.ndarray:
    """
    Compute quantile-based levels for NF4 (or any normal-float format).

    Divides the standard normal distribution into n_levels equal-probability
    regions and places a quantisation level at the centre (expected value)
    of each region. For n_levels=16, this gives the NF4 levels.

    The resulting levels are symmetric around zero because the normal
    distribution is symmetric.
    """
    # Quantile boundaries: n_levels + 1 equally spaced probability points
    quantiles  = np.linspace(0, 1, n_levels + 1)
    boundaries = scipy_norm.ppf(quantiles[1:-1])  # exclude 0 and 1 (which are -inf, +inf)

    # Level values: expected value of x within each quantile bucket
    # For a normal, E[X | a < X < b] = (phi(a) - phi(b)) / (Phi(b) - Phi(a))
    # where phi is the PDF and Phi is the CDF
    all_bounds = np.array([-np.inf] + list(boundaries) + [np.inf])
    levels = []
    for lo, hi in zip(all_bounds[:-1], all_bounds[1:]):
        # Expected value in bucket [lo, hi]
        p_lo = scipy_norm.pdf(lo) if np.isfinite(lo) else 0
        p_hi = scipy_norm.pdf(hi) if np.isfinite(hi) else 0
        prob = scipy_norm.cdf(hi) - scipy_norm.cdf(lo)
        expected = (p_lo - p_hi) / prob  # standard formula
        levels.append(expected)

    levels = np.array(levels)
    # Normalise to [-1, 1] range (NF4 convention)
    levels = levels / max(abs(levels.min()), abs(levels.max()))
    return np.sort(levels)


NF4_LEVELS = compute_nf4_levels(16)

print("NF4 quantisation levels:")
print(f"  {np.array2string(NF4_LEVELS, precision=4, separator=', ')}")
print(f"\n  Min: {NF4_LEVELS.min():.4f}")
print(f"  Max: {NF4_LEVELS.max():.4f}")
print(f"  Levels near zero (denser): {NF4_LEVELS[6]:.4f}, {NF4_LEVELS[7]:.4f}, {NF4_LEVELS[8]:.4f}, {NF4_LEVELS[9]:.4f}")
print(f"  Levels near extremes (sparser): {NF4_LEVELS[0]:.4f}, ..., {NF4_LEVELS[-1]:.4f}")

In [ ]:
# NF4 quantiser with block-wise scaling.
#
# Block-wise scaling is a key feature of NF4 as used in QLoRA:
# instead of one scale factor for the entire tensor, the tensor is split
# into blocks of 64 elements and each block gets its own scale.
# This reduces quantisation error because the effective range is smaller
# within each block than across the whole tensor.
#
# The quantisation procedure for each block:
#   1. Find the absolute maximum value in the block
#   2. Normalise the block to [-1, 1] using this maximum
#   3. Map each normalised value to the nearest NF4 level
#   4. Store the 4-bit index (0-15) and the scale factor
#
# The dequantisation procedure:
#   1. Look up the level value from the NF4 table using the 4-bit index
#   2. Multiply by the block scale factor to recover the approximate float

class NF4Quantiser:
    """
    NF4 quantiser with block-wise scaling.

    Implements the quantisation scheme from the QLoRA paper:
    Dettmers et al. (2023), Section 3.1.
    """

    def __init__(self, block_size: int = 64):
        self.block_size = block_size
        self.levels     = torch.tensor(NF4_LEVELS, dtype=torch.float32)

    def quantise(
        self,
        tensor: torch.Tensor,    # any shape; will be flattened
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Quantise a tensor to NF4.

        Returns:
            indices: int8 tensor of NF4 level indices (0-15), shape (n_elements,)
            scales:  float32 tensor of block-wise scale factors, shape (n_blocks,)
        """
        flat   = tensor.float().flatten()
        n      = flat.numel()
        # Pad to a multiple of block_size
        pad    = (self.block_size - n % self.block_size) % self.block_size
        if pad > 0:
            flat = torch.cat([flat, torch.zeros(pad)])

        blocks = flat.reshape(-1, self.block_size)   # (n_blocks, block_size)

        # Block-wise scale: absolute maximum of each block
        scales = blocks.abs().amax(dim=1, keepdim=True).clamp(min=1e-8)  # (n_blocks, 1)

        # Normalise each block to [-1, 1]
        normalised = blocks / scales   # (n_blocks, block_size)

        # Map each value to the nearest NF4 level using L1 distance
        # Shape broadcast: (n_blocks, block_size, 1) vs (1, 1, 16)
        dists   = (normalised.unsqueeze(-1) - self.levels.view(1, 1, -1)).abs()
        indices = dists.argmin(dim=-1).to(torch.int8)  # (n_blocks, block_size)

        return indices.flatten()[:n], scales.squeeze(1)

    def dequantise(
        self,
        indices: torch.Tensor,   # (n_elements,) int8
        scales:  torch.Tensor,   # (n_blocks,)  float32
        orig_shape: torch.Size,
    ) -> torch.Tensor:
        """Reconstruct float values from NF4 indices and block scales."""
        n = indices.numel()
        # Expand scales to match each element
        scales_expanded = scales.repeat_interleave(self.block_size)[:n]
        # Table lookup
        level_values = self.levels[indices.long()]
        return (level_values * scales_expanded).reshape(orig_shape)


# Compare NF4 vs INT4 uniform quantisation error on typical weight matrices
nf4_q  = NF4Quantiser(block_size=64)
int4_q = UniformQuantiser(bits=4, symmetric=True)

weight = torch.randn(512, 512)  # typical transformer weight

# NF4
idx, scales = nf4_q.quantise(weight)
nf4_approx  = nf4_q.dequantise(idx, scales, weight.shape)
nf4_rmse    = (weight - nf4_approx).pow(2).mean().sqrt().item()

# INT4 uniform
int4_approx, _, _ = int4_q.quantise_dequantise(weight)
int4_rmse = (weight - int4_approx).pow(2).mean().sqrt().item()

print("NF4 vs INT4 Uniform quantisation error on N(0,1) weights:")
print(f"  NF4  RMSE: {nf4_rmse:.6f}")
print(f"  INT4 RMSE: {int4_rmse:.6f}")
print(f"  NF4 error reduction: {(1 - nf4_rmse/int4_rmse)*100:.1f}%")
print()
print("NF4 achieves lower RMSE on normal-distributed weights because its levels")
print("are denser near zero, where most weight values concentrate.")

In [ ]:
# Visualise NF4 vs uniform INT4 level placement and error.
#
# This plot is the clearest way to understand why NF4 outperforms
# uniform quantisation for weight tensors:
#   - The weight distribution is bell-shaped (mostly values near zero)
#   - Uniform INT4 places equally spaced levels, wasting capacity at the tails
#   - NF4 clusters levels near zero, matching the weight distribution
#
# The error profile at the bottom shows that NF4's error is more uniform
# across the value range, while INT4's error is smallest in the sparse
# tail regions (where the step size is a small fraction of the value)
# and largest near zero (where many values compete for the same few levels).

x_range = np.linspace(-3.5, 3.5, 1000)

# INT4 levels for a representative weight tensor
sample_weight = torch.randn(10000)
w_min, w_max  = sample_weight.min().item(), sample_weight.max().item()
int4_scale    = (w_max - w_min) / 15
int4_levels_actual = np.array([w_min + i * int4_scale for i in range(16)])

# NF4 levels scaled to same range
nf4_levels_scaled = NF4_LEVELS * max(abs(w_min), abs(w_max))

fig, axes = plt.subplots(3, 1, figsize=(12, 11))

# Weight distribution with level positions
ax = axes[0]
ax.hist(sample_weight.numpy(), bins=80, color='lightgray', alpha=0.8,
        density=True, label='Weight distribution N(0,1)')
for level in int4_levels_actual:
    ax.axvline(level, color='#d62728', alpha=0.6, linewidth=1.0)
ax.axvline(int4_levels_actual[0], color='#d62728', alpha=0.6, linewidth=1.0,
           label='INT4 uniform levels (equal spacing)')
for level in nf4_levels_scaled:
    ax.axvline(level, color='#2ca02c', alpha=0.6, linewidth=1.0)
ax.axvline(nf4_levels_scaled[0], color='#2ca02c', alpha=0.6, linewidth=1.0,
           label='NF4 levels (denser near zero)')
ax.set_xlim(-4, 4)
ax.set_ylabel('Density', fontsize=10)
ax.set_title('Weight Distribution and Quantisation Level Placement\n'
             'NF4 concentrates levels where most weights are', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Step sizes between consecutive levels
ax = axes[1]
int4_steps = np.diff(int4_levels_actual)
nf4_steps  = np.diff(np.sort(nf4_levels_scaled))
bar_w = 0.3
x_pos = np.arange(15)
ax.bar(x_pos - bar_w/2, int4_steps, bar_w, color='#d62728', alpha=0.7, label='INT4 uniform')
ax.bar(x_pos + bar_w/2, nf4_steps,  bar_w, color='#2ca02c', alpha=0.7, label='NF4')
ax.set_xlabel('Level index', fontsize=10)
ax.set_ylabel('Step size between adjacent levels', fontsize=10)
ax.set_title('Step Size per Interval\n'
             'NF4 has smaller steps near zero (better precision where data is dense)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# RMSE on different weight distributions
ax = axes[2]
distributions = [
    ('N(0, 1)',       torch.randn(10000)),
    ('N(0, 0.5)',     torch.randn(10000) * 0.5),
    ('Uniform[-1,1]', torch.rand(10000) * 2 - 1),
    ('Laplace(0,1)',  torch.distributions.Laplace(0, 1).sample((10000,))),
]

labels, nf4_errors, int4_errors = [], [], []
for label, w in distributions:
    idx, sc = nf4_q.quantise(w)
    nf4_r   = nf4_q.dequantise(idx, sc, w.shape)
    nf4_rmse_d = (w - nf4_r).pow(2).mean().sqrt().item()

    int4_r, _, _ = int4_q.quantise_dequantise(w)
    int4_rmse_d  = (w - int4_r).pow(2).mean().sqrt().item()

    labels.append(label)
    nf4_errors.append(nf4_rmse_d)
    int4_errors.append(int4_rmse_d)

x_pos = np.arange(len(labels))
ax.bar(x_pos - 0.2, int4_errors, 0.4, color='#d62728', alpha=0.8, label='INT4 uniform')
ax.bar(x_pos + 0.2, nf4_errors,  0.4, color='#2ca02c', alpha=0.8, label='NF4')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('RMSE', fontsize=10)
ax.set_title('Quantisation RMSE by Weight Distribution\n'
             'NF4 wins on normal/laplace; INT4 wins on uniform (unsurprisingly)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('NF4 vs INT4 Uniform Quantisation Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/09_nf4_vs_int4.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Post-Training Quantisation with bitsandbytes

In [ ]:
# Post-training quantisation (PTQ) with bitsandbytes.
#
# bitsandbytes implements quantisation as a transparent layer wrapping
# HuggingFace models. Passing load_in_8bit=True or load_in_4bit=True
# to from_pretrained() automatically quantises all linear layers while
# loading, without any changes to the model architecture or training code.
#
# The quantisation configuration controls:
#   load_in_4bit:          use 4-bit weights (NF4 or FP4)
#   bnb_4bit_quant_type:   'nf4' or 'fp4' (NF4 is better for most models)
#   bnb_4bit_compute_dtype: the dtype used for the dequant-multiply-requant
#                           computation (bf16 is recommended for accuracy)
#   bnb_4bit_use_double_quantisation: quantise the scale factors themselves
#                           (saves 0.37 bits/param, see QLoRA paper Section 3.2)
#
# We load GPT-2 in fp16 and int8 to compare memory and output quality.
# Note: load_in_8bit requires a CUDA GPU. On CPU we measure the from-scratch
# quantisation impact instead.

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'gpt2'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

# Helper: measure model memory usage
def model_memory_mb(model: nn.Module) -> float:
    """Estimate model parameter memory in MB."""
    total_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    return total_bytes / (1024 ** 2)


# Load fp16 model (reference)
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16
)
fp16_mem = model_memory_mb(model_fp16)

print(f"GPT-2 memory footprint:")
print(f"  fp16:     {fp16_mem:.1f} MB")
print(f"  int8 (theoretical):    {fp16_mem/2:.1f} MB  (2x compression)")
print(f"  nf4 (theoretical):     {fp16_mem/4:.1f} MB  (4x compression)")
print()

# Configure 4-bit NF4 quantisation (load_in_4bit=True requires CUDA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

if torch.cuda.is_available():
    model_4bit = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto',
    )
    # bitsandbytes stores quantised weights separately; the reported
    # parameter count in bytes is lower than fp16
    print("NF4 model loaded on GPU.")
    print(f"  GPU memory (allocated): {torch.cuda.memory_allocated() / 1e6:.0f} MB")
else:
    print("No GPU available -- NF4 quantisation requires CUDA for bitsandbytes.")
    print("The BitsAndBytesConfig is defined above and would be passed to")
    print("from_pretrained() on a CUDA system.")
    print()
    print("Memory savings summary (estimated for any model):")
    print(f"  {'Format':<8} {'Memory (MB)':>14} {'Compression':>14}")
    for label, factor in [('fp32', 1), ('fp16', 2), ('int8', 4), ('nf4', 8)]:
        compressed_mb = fp16_mem * 2 / factor  # fp32 baseline / compression
        print(f"  {label:<8} {compressed_mb:>14.1f} {factor:>12}x")

In [ ]:
# Quantisation accuracy measurement on GPT-2.
#
# We measure accuracy using perplexity on a small held-out text sample.
# Perplexity is the standard language model quality metric (see Notebook 06.01).
# Lower perplexity means the model assigns higher probability to the text,
# indicating better language modelling quality.
#
# We compare three precision levels:
#   fp32:  reference, highest quality
#   fp16:  standard deployment precision
#   int8:  our from-scratch quantised model (applied weight-by-weight)
#
# We implement model-level quantisation using the UniformQuantiser from
# section 3, replacing each weight matrix with its quantised-dequantised
# approximation. This is 'simulated quantisation' (also called 'fake
# quantisation') -- the computation still runs in fp32 but with weights
# that have been rounded to the nearest integer level.

def apply_weight_quantisation(
    model: nn.Module,
    bits:  int = 8,
) -> nn.Module:
    """
    Apply simulated post-training quantisation to all linear layers.

    Replaces each weight matrix in-place with its quantised-dequantised
    approximation. The model continues to run in floating point but with
    reduced effective precision in the weights.
    """
    import copy
    q_model = copy.deepcopy(model)
    quantiser = UniformQuantiser(bits=bits, symmetric=True, per_channel=True)

    for name, module in q_model.named_modules():
        if isinstance(module, nn.Linear):
            with torch.no_grad():
                approx, _, _ = quantiser.quantise_dequantise(module.weight.data)
                module.weight.data = approx

    return q_model


def compute_perplexity(model: nn.Module, tokenizer, text: str) -> float:
    """Compute perplexity of a text string under a causal language model."""
    model.eval()
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    input_ids = inputs['input_ids']

    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss    = outputs.loss

    return torch.exp(loss).item()


# Evaluation text: diverse sample for a fair perplexity estimate
EVAL_TEXT = (
    "The transformer architecture, introduced in the paper Attention Is All You Need, "
    "relies on self-attention mechanisms to process sequences of tokens in parallel. "
    "Unlike recurrent neural networks, transformers do not have an inherent notion of "
    "token order; instead, positional encodings are added to the token embeddings to "
    "provide sequence position information. The self-attention operation computes a "
    "weighted sum of value vectors, where the weights are determined by the dot product "
    "similarity between query and key vectors. This allows each token to attend to any "
    "other token in the sequence regardless of their distance, making transformers "
    "particularly effective at capturing long-range dependencies."
)

model_fp32 = AutoModelForCausalLM.from_pretrained(MODEL_NAME).eval()
model_int8 = apply_weight_quantisation(model_fp32, bits=8)
model_int4 = apply_weight_quantisation(model_fp32, bits=4)

results = [
    ('fp32 (reference)',   model_fp32),
    ('int8 (simulated)',   model_int8),
    ('int4 (simulated)',   model_int4),
]

print("Perplexity comparison (lower = better quality):")
print()
reference_ppl = None
for label, m in results:
    ppl = compute_perplexity(m, tokenizer, EVAL_TEXT)
    if reference_ppl is None:
        reference_ppl = ppl
        delta = '--'
    else:
        delta = f"+{((ppl/reference_ppl)-1)*100:.1f}%"
    mem  = model_memory_mb(m)
    print(f"  {label:<22}  perplexity={ppl:.2f}  degradation={delta:<10}  memory={mem:.1f} MB")

## 6. Engineering Notes

**Quantisation strategy decision guide**

| Goal | Recommended approach |
|------|---------------------|
| Minimal accuracy loss, fastest inference | fp16 or bf16 (no quantisation) |
| Fit 7B model on single 24 GB GPU | INT8 with bitsandbytes (`load_in_8bit=True`) |
| Fit 13B model on single 24 GB GPU | NF4 with bitsandbytes (`load_in_4bit=True, bnb_4bit_quant_type='nf4'`) |
| Fastest 4-bit inference (batched) | GPTQ (offline calibration, faster than bitsandbytes at batch size > 1) |
| Edge deployment (no CUDA) | GGUF/llama.cpp with Q4_K_M quantisation |
| Highest accuracy at 4-bit | AWQ (activation-aware weight quantisation) |

**What to quantise and what not to**

Not all layers are equally tolerant of quantisation error:
- **Attention weight matrices** (Q, K, V, O): moderately sensitive; INT8 is safe, INT4 may degrade
- **Feed-forward layers**: less sensitive; INT4 is usually fine
- **Embedding layer**: sensitive; keep in fp16 or fp32
- **Layer norm parameters**: very sensitive (small tensors, cheap to keep in fp16)
- **LM head**: keep in fp16 for numerical stability in the final projection

## 7. Exercises

1. **Per-channel vs per-tensor**: Compare quantisation RMSE for INT8 symmetric quantisation applied per-tensor vs per-output-channel on a 768x3072 feed-forward weight matrix. How much does per-channel improve accuracy?

2. **Calibration data sensitivity**: Apply INT8 quantisation with absmax scaling (scale = max absolute value). Then try percentile-based scaling (scale = 99.9th percentile). Measure the effect on perplexity. Does outlier-robust scaling help?

3. **Layer sensitivity analysis**: Quantise one transformer layer at a time (leaving all others in fp32) and measure the perplexity after each. Which layers are most sensitive to quantisation? Does the pattern match the intuition about embedding and layer norm?

4. **GPTQ vs bitsandbytes**: Load the same model with `load_in_4bit=True` (bitsandbytes) and with a GPTQ-quantised version from the HuggingFace hub. Measure perplexity and inference throughput at batch sizes 1, 4, and 16. When does GPTQ's faster kernel start to dominate?

5. **Double quantisation benefit**: Implement double quantisation from scratch: quantise the NF4 block scales themselves using 8-bit uniform quantisation. Measure the additional memory saving and the accuracy impact.

---

## 8. References

- [Dettmers et al. (2023) -- QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)
- [Frantar et al. (2022) -- GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers](https://arxiv.org/abs/2210.17323)
- [Lin et al. (2024) -- AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration](https://arxiv.org/abs/2306.00978)
- [Dettmers et al. (2022) -- LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale](https://arxiv.org/abs/2208.07339)
- [bitsandbytes Documentation](https://huggingface.co/docs/bitsandbytes)